# 01 - Minimizing cost

**ICSSI 2026, Using LLMs via the API**

Cost is the difference between a demo that runs once and a pipeline you can run over a
whole corpus. We will work on a real task, classifying paper abstracts into research
fields, and learn five strategies. First, count tokens before you send so you can price a
job up front. Second, read the usage to see input and output cost on every call. Third,
use model tiering, because Opus, Sonnet, and Haiku do the same task at very different
prices. Fourth, use prompt caching to reuse a big shared context for about a tenth of the
cost. Fifth, use the Batch API to run non-urgent bulk jobs at half price.

In [ ]:
# fetch the repo files (claude_kit.py and tutorial_data/) when running in Colab
import os, sys, subprocess
REPO_URL = "https://github.com/LarremoreLab/icssi-2026-hackathon.git"  # update after you push to GitHub

if "google.colab" in sys.modules and not os.path.exists("claude_kit.py"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git"))
print("working directory:", os.getcwd())
assert os.path.exists("claude_kit.py"), (
    "claude_kit.py was not found. Run this notebook from inside the cloned repo folder, "
    "or set REPO_URL above and run this cell again in Colab."
)

In [ ]:
# bootstrap: works in Google Colab and locally
import os, sys, subprocess

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

try:
    import anthropic
except ImportError:
    pip_install("anthropic")
    import anthropic

# find the api key: env var first, then a Colab secret, then a local .env file
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv
    except ImportError:
        pip_install("python-dotenv")
        from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "No ANTHROPIC_API_KEY found. Running locally, copy .env.example to .env and add your "
    "key. In Colab, add it in the Secrets panel (the key icon), named ANTHROPIC_API_KEY."
)
print("anthropic version:", anthropic.__version__, "| API key loaded: OK")
MODEL = "claude-sonnet-4-6"

## Load the data and the helper kit

The file `tutorial_data/field_taxonomy.md` is our classification codebook, which lists the field
codes and when to use each one. The file `tutorial_data/abstracts.json` holds a small set of
science-of-science abstracts. The module `claude_kit.py` gives us a `CostTracker` so we
can watch spend add up.

In [ ]:
import json
from claude_kit import ClaudeClient, CostTracker

taxonomy = open("tutorial_data/field_taxonomy.md").read()
abstracts = json.load(open("tutorial_data/abstracts.json"))

print(f"codebook: {len(taxonomy):,} chars   abstracts: {len(abstracts)}")
print("first abstract:", abstracts[0]["title"])

## Strategy 1: count tokens before you spend

The method `client.messages.count_tokens(...)` returns the input-token count for a request
without running it. Use it to estimate the cost of a big job before you launch it.

In [ ]:
client = anthropic.Anthropic()

first_abstract = abstracts[0]
messages = [{"role": "user", "content": "Classify this abstract.\n\n" + first_abstract["abstract"]}]

token_count = client.messages.count_tokens(
    model=MODEL, system=taxonomy, messages=messages,
)
print("input tokens for one classification:", token_count.input_tokens)

# price the whole job, input only, at Opus rates for 100k abstracts
input_cost_per_call = token_count.input_tokens * 5.0 / 1_000_000
print(f"about ${input_cost_per_call:.5f} input per abstract, "
      f"so about ${input_cost_per_call * 100_000:,.2f} to classify 100k (input only)")

## Strategy 2: read the response `usage` metadata

Let us actually classify one abstract. We use the `CostTracker` from `claude_kit`, which
records the usage of every call and converts it to dollars.

In [ ]:
tracker = CostTracker(budget_usd=0.50)   # warn us if we cross fifty cents
kit = ClaudeClient(model=MODEL, max_tokens=20, tracker=tracker)

prediction = kit.ask(
    "Classify this abstract. Reply with ONLY the field code.\n\n" + first_abstract["abstract"],
    system=taxonomy,
    label="classify-opus",
)
print("predicted field code:", prediction.strip())
print("recorded:", tracker.records[-1])

## Strategy 3: choose the appropriate model

The same prompt on a cheaper model can be plenty good for an easy task. Opus 4.8 is the
most capable. Sonnet 4.6 is about forty percent cheaper. Haiku 4.5 is roughly five times
cheaper than Opus. Classification is exactly the kind of task where a small model often
suffices, so try the cheapest one that is accurate enough.

In [ ]:
models = ["claude-opus-4-8", "claude-sonnet-4-6", "claude-haiku-4-5"]
print(f"{'model':<20}{'label':<10}{'cost $':>12}")
for model in models:
    prediction = kit.ask(
        "Classify this abstract. Reply with ONLY the field code.\n\n" + first_abstract["abstract"],
        system=taxonomy, model=model, label=f"classify-{model}",
    )
    print(f"{model:<20}{prediction.strip():<10}{tracker.records[-1]['cost']:>12.6f}")

## Strategy 4: prompt caching

When many requests share a big identical prefix, here the codebook, you can cache it. The
first call writes the cache at about 1.25 times cost, and later calls read it for about a
tenth of the cost.

Caching is a prefix match, which means the cached part must be byte-for-byte identical and
must come first. We put the codebook plus a few worked examples in the system field with a
`cache_control` marker, and we vary only the user's abstract.

Each model has a minimum cacheable prefix. Opus 4.8 and Sonnet 4.6 need 1024 tokens while Haiku 4.5 needs 4096. We use Sonnet here, together with a larger codebook prefix, so the prefix clears the
bar. Watch `cache_creation_input_tokens` on the first call and `cache_read_input_tokens`
on the second.

In [ ]:
# build a large, stable prefix: the codebook plus labeled examples
examples = "\n\n".join(
    f"EXAMPLE\nAbstract: {item['abstract']}\nField code: (one of the codes above)"
    for item in abstracts
)
cached_system_prompt = [{
    "type": "text",
    "text": taxonomy + "\n\n# Worked examples\n" + examples,
    "cache_control": {"type": "ephemeral"},
}]

def classify_cached(abstract, model="claude-sonnet-4-6"):
    response = client.messages.create(
        model=model, max_tokens=20, system=cached_system_prompt,
        messages=[{"role": "user",
                   "content": "Classify this abstract. Reply with ONLY the code.\n\n" + abstract}],
    )
    tracker.add(model, response.usage, label="cached-classify")
    return response

first_response = classify_cached(abstracts[1]["abstract"])
print("call 1  cache_creation:", first_response.usage.cache_creation_input_tokens,
      " cache_read:", first_response.usage.cache_read_input_tokens)

second_response = classify_cached(abstracts[2]["abstract"])
print("call 2  cache_creation:", second_response.usage.cache_creation_input_tokens,
      " cache_read:", second_response.usage.cache_read_input_tokens, " (reused the codebook)")

## Strategy 5: the Batch API at fifty percent off

For non-urgent bulk jobs you can submit many independent requests at once and pay half the
standard token price. Results come back asynchronously. Most batches finish in minutes, and
the maximum is 24 hours. This is the right tool for classifying or extracting over a whole
corpus when you do not need answers right now.

We classify every sample abstract on Haiku in one batch.

**Notes:**
- Use meaningful values for the `custom_id` field of your batch requests so that you can link the batch responses back to the right job. The order of jobs returned from the Batch API is not guaranteed!
- You can use prompt caching with the Batch API to stack savings.

In [ ]:
import time
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

cached_system_prompt = [
    {
        "type": "text",
        "text": taxonomy + "\n\n# Worked examples\n" + examples,
        "cache_control": {"type": "ephemeral"},
    }
]
batch_requests = [
    Request(
        custom_id=item["id"],
        params=MessageCreateParamsNonStreaming(
            model="claude-haiku-4-5",
            max_tokens=20,
            system=cached_system_prompt,
            messages=[
                {
                    "role": "user",
                    "content": "Classify this abstract. Reply with ONLY the code.\n\n"
                    + item["abstract"],
                }
            ],
        ),
    )
    for item in abstracts
]

batch = client.messages.batches.create(requests=batch_requests)
print("submitted batch:", batch.id)

In [ ]:
from claude_kit import cost_of_usage

# poll until done, which is usually quick for a tiny batch
while True:
    batch_status = client.messages.batches.retrieve(batch.id)
    if batch_status.processing_status == "ended":
        break
    print("status:", batch_status.processing_status, "...")
    time.sleep(10)

standard_cost = 0.0
for result in client.messages.batches.results(batch.id):
    if result.result.type == "succeeded":
        message = result.result.message
        label_text = "".join(block.text for block in message.content if block.type == "text")
        standard_cost += cost_of_usage(message.usage, "claude-haiku-4-5")
        tracker.add_response(model="claude-haiku-4-5", response=message, label="haiku_batch_api_abstract_classification")   # add to our running tracker
        print(f"{result.custom_id}: {label_text.strip()}")

print(f"\nstandard token cost would be ${standard_cost:.6f}; "
      f"the Batch API bills half, about ${standard_cost / 2:.6f}")

## Where the savings are

For a classification or extraction job, the cheapest strategies, in order, are these.
First, pick the smallest model that passes your accuracy bar, which is often Haiku or
Sonnet. Second, cache any large shared context across calls. Third, keep outputs tiny by
asking for a code rather than an essay, because output tokens cost the most. Fourth, for
non-urgent bulk jobs, use the Batch API at fifty percent off.

Here is the running total for the interactive calls we made. The batch above is billed
separately at half price, as shown in its own output.

In [ ]:
tracker.report()